Khushi Khatri BEB222 Experiment 8
Aim: Implement Named Entity Recognizer on the given dataset.

Theory
Named Entity Recognition (NER) is the task of locating and classifying
"named entities" in text into predefined categories such as PERSON,
ORGANIZATION, GPE (geo-political entity — countries, cities, states),
LOCATION, DATE, TIME, MONEY, and PERCENT. Where POS tagging (Experiment 6)
labels each word's grammatical role and chunking (Experiment 7) groups
words into flat phrases, NER goes a step further and assigns real-world
meaning to specific phrases.

1. Why NER for this project?
Airbnb reviews frequently mention named entities that matter for
understanding the review: landmarks and neighbourhoods (GPE/LOCATION,
e.g. "Eiffel Tower", "Montmartre"), the host's name (PERSON), dates of
stay (DATE), and prices (MONEY). Extracting these lets us go beyond
generic sentiment scoring towards understanding *what* or *who* a review
is actually talking about — the same aspect-based idea introduced with
noun-phrase chunking in Experiment 7, but now with real-world entity
categories instead of just grammatical phrase types.

In [5]:
import pandas as pd
from collections import Counter
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')
from nltk.tokenize import word_tokenize
from nltk import ne_chunk
print('Setup complete.')

[nltk_data] Downloading package punkt to /home/computer/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /home/computer/nltk_data...


Setup complete.


[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.
[nltk_data] Downloading package words to /home/computer/nltk_data...
[nltk_data]   Package words is already up-to-date!


In [6]:
df = pd.read_csv('Balanced_Airbnb_Reviews_Dataset.csv')
print('Shape:', df.shape)
df[['review_id', 'review_text']].head()

Shape: (15000, 42)


,review_id,review_text
0,369314882,Amazing stay! The place felt very cozy for 4 g...
1,490116563,It was okay for the price. Location in XIII Au...
2,582235668,Loved every minute of it. Our superhost was su...
3,68054683,Decent stay overall. Some things could be impr...
4,248483824,Reasonable for a short trip. Location in Long ...


In [7]:
def ner_text(text):
    """Tokenize -> POS tag -> NER chunk. Returns an nltk Tree."""
    if not isinstance(text, str) or text.strip() == '':
        return None
    tokens = word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    return ne_chunk(tagged)
sample_text = df['review_text'].iloc[0]
print('Original Text:\n', sample_text, '\n')
sample_tree = ner_text(sample_text)
print(sample_tree)

Original Text:
 Amazing stay! The place felt very cozy for 4 guests. Check-in was smooth and the amenities were exactly what we needed. 

(S
  Amazing/VBG
  stay/NN
  !/.
  The/DT
  place/NN
  felt/VBD
  very/RB
  cozy/JJ
  for/IN
  4/CD
  guests/NNS
  ./.
  Check-in/NNP
  was/VBD
  smooth/JJ
  and/CC
  the/DT
  amenities/NNS
  were/VBD
  exactly/RB
  what/WP
  we/PRP
  needed/VBD
  ./.)


In [8]:
sample_tree.pretty_print()

                                                                                         S                                                                                                              
      ___________________________________________________________________________________|____________________________________________________________________________________________________________   
Amazing/VBG stay/NN !/. The/DT place/NN felt/VBD very/RB cozy/JJ for/IN 4/CD guests/NNS ./. Check-in/NNP was/VBD smooth/JJ and/CC the/DT amenities/NNS were/VBD exactly/RB what/WP we/PRP needed/VBD ./.



In [9]:
def extract_entities(tree):
    """Return a list of (entity_text, label) pairs from an NE-chunked tree."""
    if tree is None:
        return []
    entities = []
    for subtree in tree.subtrees():
        if subtree.label() != 'S':
            entity_text = ' '.join(word for word, tag in subtree.leaves())
            entities.append((entity_text, subtree.label()))
    return entities


entities = extract_entities(sample_tree)
print('Named Entities:', entities)

Named Entities: []


In [10]:
demo_text = "We stayed near the Eiffel Tower in Paris and the host, Marie, was wonderful."
demo_tree = ner_text(demo_text)
print(demo_tree)
print()
print('Extracted entities:', extract_entities(demo_tree))

(S
  We/PRP
  stayed/VBD
  near/IN
  the/DT
  (ORGANIZATION Eiffel/NNP Tower/NNP)
  in/IN
  (GPE Paris/NNP)
  and/CC
  the/DT
  host/NN
  ,/,
  (PERSON Marie/NNP)
  ,/,
  was/VBD
  wonderful/JJ
  ./.)

Extracted entities: [('Eiffel Tower', 'ORGANIZATION'), ('Paris', 'GPE'), ('Marie', 'PERSON')]


In [11]:
def ner_row(text):
    tree = ner_text(text)
    return extract_entities(tree)


sample_df = df.head(1000).copy()
sample_df['entities'] = sample_df['review_text'].apply(ner_row)

sample_df[['review_id', 'review_text', 'entities']].head(10)

,review_id,review_text,entities
0,369314882,Amazing stay! The place felt very cozy for 4 g...,[]
1,490116563,It was okay for the price. Location in XIII Au...,"[(XIII Aurelia, ORGANIZATION)]"
2,582235668,Loved every minute of it. Our superhost was su...,"[(Great, GPE), (Venustiano Carranza, GPE), (Me..."
3,68054683,Decent stay overall. Some things could be impr...,"[(Decent, GPE)]"
4,248483824,Reasonable for a short trip. Location in Long ...,"[(Long Island City, GPE)]"
5,155617131,Decent stay overall. It served its purpose for...,"[(Decent, GPE), (Paris, GPE)]"
6,710244614,"Nothing special, but fine. Our superhost was p...",[]
7,299174484,We had a rough experience. The location in Enc...,"[(Paris, GPE)]"
8,22136604,Perfect for our trip. Check-in was smooth and ...,"[(Perfect, GPE), (Great, GPE), (Paris, GPE)]"
9,469473761,Would not recommend. The private room in house...,[]


In [12]:
all_entities = [ent for row in sample_df['entities'] for ent in row]

entity_text_counts = Counter(text for text, label in all_entities)
entity_label_counts = Counter(label for text, label in all_entities)

print('Top 15 Named Entities across dataset:')
for text, count in entity_text_counts.most_common(15):
    print(text, '->', count)

print('\nEntity Label Distribution:')
for label, count in entity_label_counts.most_common():
    print(f'{label}: {count}')

Top 15 Named Entities across dataset:
Great -> 144
Rome -> 94
Paris -> 86
New York -> 76
Decent -> 70
Centro Storico -> 52
Wonderful -> 48
Mexico City -> 45
Perfect -> 42
Sydney -> 41
Rio -> 26
Janeiro -> 26
Bangkok -> 22
Cuauhtemoc -> 20
Cape Town -> 17

Entity Label Distribution:
GPE: 962
PERSON: 122
ORGANIZATION: 31
LOCATION: 7
GSP: 4


In [13]:
sample_df.to_csv('NER_Airbnb_Reviews.csv', index=False)
print('Saved to NER_Airbnb_Reviews.csv')
print(sample_df.shape)

Saved to NER_Airbnb_Reviews.csv
(1000, 43)
